# Fine-Tuning Llama 3 on a Medical Dataset (Google Colab)

Based on [DataCamp's Llama 3 Fine-Tuning Tutorial](https://www.datacamp.com/tutorial/llama3-fine-tuning-locally).

This notebook fine-tunes **Llama 3 8B Chat** on the [ai-medical-chatbot](https://huggingface.co/datasets/ruslanmv/ai-medical-chatbot) dataset using QLoRA, then saves and pushes the adapter to Hugging Face Hub.

**Setup:** Runtime → Change runtime type → T4 GPU (or better)

**Prerequisites:**
- Accept [Meta Llama 3.1 license](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct)
- Create a [Hugging Face token](https://huggingface.co/settings/tokens)

## 1. Install Dependencies

In [ ]:
%%capture
!pip install -U transformers datasets accelerate peft trl bitsandbytes huggingface_hub

## 2. Login to Hugging Face Hub

In [ ]:
from huggingface_hub import login

# Paste your Hugging Face token (get it from https://huggingface.co/settings/tokens)
# Accept Llama 3.1 license at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
login(token="hf_YOUR_TOKEN_HERE")

## 3. Configuration

In [ ]:
# Model and dataset config - load from Hugging Face Hub
base_model = "meta-llama/Llama-3.1-8B-Instruct"
dataset_name = "ruslanmv/ai-medical-chatbot"
new_model = "YOUR_USERNAME/llama-3-8b-chat-doctor"  # Change to your HF username

attn_implementation = "eager"  # Use "sdpa" for newer PyTorch if supported

## 4. Load Model and Tokenizer

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, setup_chat_format
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# QLoRA config - 4-bit quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load model from Hugging Face Hub
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=attn_implementation,
    trust_remote_code=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model)

# Setup chat format (ChatML template)
model, tokenizer = setup_chat_format(model, tokenizer)

## 5. Add LoRA Adapter

In [ ]:
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["up_proj", "down_proj", "gate_proj", "k_proj", "q_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, peft_config)

## 6. Load and Prepare Dataset

In [ ]:
from datasets import load_dataset

# Load dataset from Hugging Face Hub
dataset = load_dataset(dataset_name, split="all")
dataset = dataset.shuffle(seed=65).select(range(1000))  # Use 1000 samples for quick demo

def format_chat_template(row):
    row_json = [
        {"role": "user", "content": row["Patient"]},
        {"role": "assistant", "content": row["Doctor"]},
    ]
    row["text"] = tokenizer.apply_chat_template(row_json, tokenize=False)
    return row

dataset = dataset.map(format_chat_template, num_proc=4)

# Show sample
print(dataset["text"][3][:500] + "...")

In [ ]:
dataset = dataset.train_test_split(test_size=0.1)

## 7. Training

In [ ]:
training_arguments = TrainingArguments(
    output_dir=new_model,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    num_train_epochs=1,
    evaluation_strategy="steps",
    eval_steps=0.2,
    logging_steps=1,
    warmup_steps=10,
    logging_strategy="steps",
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    group_by_length=True,
    report_to="none",  # Set to "wandb" if you want W&B logging
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    max_seq_length=512,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_arguments,
    packing=False,
)

trainer.train()

## 8. Evaluation

In [ ]:
model.config.use_cache = True

messages = [{"role": "user", "content": "Hello doctor, I have bad acne. How do I get rid of it?"}]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to("cuda")

outputs = model.generate(**inputs, max_length=150, num_return_sequences=1)
text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Model response:")
print(text.split("assistant")[1] if "assistant" in text else text)

## 9. Save and Push to Hugging Face Hub

In [ ]:
# Save adapter locally
trainer.model.save_pretrained(new_model)
tokenizer.save_pretrained(new_model)

# Push adapter to Hugging Face Hub
trainer.model.push_to_hub(new_model, use_temp_dir=False)
tokenizer.push_to_hub(new_model, use_temp_dir=False)

print(f"Adapter pushed to https://huggingface.co/{new_model}")

## 10. Optional: Merge Adapter with Base Model

Run this section if you want to merge the LoRA adapter with the base model and push the full model to the Hub. **Note:** The full model is ~16GB and may exceed Colab memory limits.

In [ ]:
from peft import PeftModel

# Reload base model (full precision for merge)
base_model_reload = AutoModelForCausalLM.from_pretrained(
    base_model,
    return_dict=True,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
base_model_reload, tokenizer = setup_chat_format(base_model_reload, tokenizer)

# Merge adapter with base model
merged_model = PeftModel.from_pretrained(base_model_reload, new_model)
merged_model = merged_model.merge_and_unload()

# Save and push merged model
merged_model.save_pretrained("llama-3-8b-chat-doctor-merged")
tokenizer.save_pretrained("llama-3-8b-chat-doctor-merged")

# Push to Hub (optional)
# merged_model.push_to_hub("YOUR_USERNAME/llama-3-8b-chat-doctor-merged", use_temp_dir=False)
# tokenizer.push_to_hub("YOUR_USERNAME/llama-3-8b-chat-doctor-merged", use_temp_dir=False)

## 11. Optional: Push GGUF to Hugging Face (Quantized)

To convert to GGUF and quantize for local use (Jan, Ollama, etc.), you can use the [gguf-my-repo](https://huggingface.co/spaces/ggml-org/gguf-my-repo) Hugging Face Space: provide your merged model repo ID and it will convert and quantize automatically.

Alternatively, clone llama.cpp and run the conversion scripts locally.